In [1]:
import tomllib
with open('config.toml', 'rb') as f:
    config = tomllib.load(f)
config

{'base_url': 'http://localhost:1234/v1',
 'model': 'openai/gpt-oss-20b',
 'big_model': 'openai/gpt-oss-120b',
 'unthinking': 'meta/llama-3.3-70b',
 'embedding_model': 'text-embedding-embeddinggemma-300m-qat',
 'api_key': 'local'}

# 01 · RAG Under the Covers

## What RAG actually is

When language modeling was gaining traction, due to the scaling of Transformer architectures, "hallucinations" were common and why a model would "say" something wasn't clear, RAG was developed to address this. 
https://dl.acm.org/doi/abs/10.5555/3495724.3496517

Typically it involved 2 models an embedding model trained for indexing and search, and then a language model to apply that information in an NLP context.

Today RAG is usually used as a **prompt-construction pipeline**:

```
corpus
  → chunk(corpus)
  → db = embed(chunks)
query
  → embed(query)             # dense vector
  → search(vector, db)       # top-k chunks
  → prompt = query + search
  → LLM(prompt)              # grounded generation
```

The LLM has **no special retrieval ability** — it only sees text. RAG lives entirely in the scaffolding.

This is often how things like "Memory" are implemented, and is the basis for Claude "Skills".

## Manually make a RAG system

### Step 1 — Build a toy corpus & chunk it

In [2]:
CORPUS = {
    "python_gc": """
        Python uses reference counting as its primary garbage collection mechanism.
        Every object has a reference count; when it drops to zero the memory is freed immediately.
        Cyclic references (A->B->A) are handled by a separate cyclic GC that runs periodically.
        The cyclic GC is generational: objects that survive collection are promoted to older generations
        which are collected less frequently. You can interact with it via the `gc` module.
    """,
    "rust_ownership": """
        Rust enforces memory safety at compile time through the ownership system.
        Each value has exactly one owner; ownership can be moved or borrowed.
        A borrow can be either shared (&T, many readers) or exclusive (&mut T, one writer).
        These rules are checked by the borrow checker and eliminate use-after-free
        and data-race bugs without a garbage collector.
    """,
    "go_gc": """
        Go uses a concurrent tri-color mark-and-sweep garbage collector.
        The GC runs concurrently with the application to minimize stop-the-world pauses.
        Go 1.5 introduced a hybrid barrier that keeps pause times under 1ms.
        The GOGC environment variable controls the GC trigger: it sets the heap growth
        percentage before a collection cycle starts (default 100).
    """,
}

### How to split documents?
Lot of different thoughts and research, but character splitting is good enough normally.

Some starting places to look more into it:
- https://arxiv.org/pdf/2312.06648
- https://medium.com/@shravankoninti/mastering-rag-a-deep-dive-into-text-splitting-fafeffdcc00d

Excersice
---
Can you make proposition retrieval by make a propositizer using an LLM? https://arxiv.org/pdf/2312.06648

In [3]:
def chunk_text(text: str, chunk_size: int = 150, overlap: int = 30) -> list[str]:
    """Simple character-level chunker with overlap."""
    text = ' '.join(text.split())
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return chunks

all_chunks = []
for doc_id, text in CORPUS.items():
    for i, chunk in enumerate(chunk_text(text)):
        all_chunks.append({"id": f"{doc_id}_{i}", "text": chunk})

print(f"Total chunks: {len(all_chunks)}")
for c in all_chunks:
    print(f"  [{c['id']}] {c['text'][:80]}...")

Total chunks: 10
  [python_gc_0] Python uses reference counting as its primary garbage collection mechanism. Ever...
  [python_gc_1] drops to zero the memory is freed immediately. Cyclic references (A->B->A) are h...
  [python_gc_2]  periodically. The cyclic GC is generational: objects that survive collection ar...
  [python_gc_3] e collected less frequently. You can interact with it via the `gc` module....
  [rust_ownership_0] Rust enforces memory safety at compile time through the ownership system. Each v...
  [rust_ownership_1] n be moved or borrowed. A borrow can be either shared (&T, many readers) or excl...
  [rust_ownership_2] are checked by the borrow checker and eliminate use-after-free and data-race bug...
  [go_gc_0] Go uses a concurrent tri-color mark-and-sweep garbage collector. The GC runs con...
  [go_gc_1] ze stop-the-world pauses. Go 1.5 introduced a hybrid barrier that keeps pause ti...
  [go_gc_2] able controls the GC trigger: it sets the heap growth percentage befor

### Step 2 — Embed everything

Why use a separate embedding model?
- Training guarentees that vector comparisions are valid.
- Embedding models are faster/easier to run.

If you want to dig more into these types of models: https://sbert.net/docs/sentence_transformer/pretrained_models.html

In [4]:
import openai, numpy as np, os

client = openai.OpenAI(base_url=config['base_url'], api_key=config['api_key'])

# To use a real embeddings endpoint (e.g. vLLM with an embedding model):
def embed_real(texts):
    resp = client.embeddings.create(model=config['embedding_model'], input=texts)
    import numpy as np
    vecs = np.array([d.embedding for d in resp.data], dtype='float32')
    vecs /= np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs

embed = embed_real

texts = [c['text'] for c in all_chunks]
index = embed(texts)
print(f'Index shape: {index.shape}   (n_chunks × embedding_dim)')


Index shape: (10, 768)   (n_chunks × embedding_dim)


### Step 3 — Retrieval

Here we do brute-force cosine similarity, usually used when vectors are thought of as directions of "concepts". There is fair amount of research exploring different distance metrics. 

Since our vectors are L2-normalised: `cos(θ) = v₁ · v₂` — just a dot product.

We are using dense vectors here, but there are sparse vector models, and different search strategies.
Qdrant has a good starting place to look at why you might use them: https://qdrant.tech/documentation/search/search/

In [5]:
def retrieve(query: str, k: int = 3, threshold: float = 0.0) -> list[dict]:
    q_vec = embed([query])[0]
    scores = index @ q_vec                    # (N,) cosine similarities
    top_k = np.argsort(scores)[::-1][:k]
    return [
        {"score": float(scores[i]), **all_chunks[i]}
        for i in top_k
        if scores[i] >= threshold
    ]

query = "How does Python handle cyclic references?"
for r in retrieve(query):
    print(f"score={r['score']:.3f}  [{r['id']}]  {r['text'][:100]}")

score=0.681  [python_gc_0]  Python uses reference counting as its primary garbage collection mechanism. Every object has a refer
score=0.642  [python_gc_1]  drops to zero the memory is freed immediately. Cyclic references (A->B->A) are handled by a separate
score=0.531  [python_gc_3]  e collected less frequently. You can interact with it via the `gc` module.


### Step 4 — Prompt construction

This is the entire secret of RAG: **inject retrieved chunks as context**. How this is done and the best way to do this varies between frameworks and models you are using.

In [6]:
def build_rag_prompt(query: str, chunks: list[dict]) -> str:
    if not chunks:
        context_block = "[No relevant context found]"
    else:
        context_block = "\n\n".join(
            f"[{c['id']} | score={c['score']:.3f}]\n{c['text']}" for c in chunks
        )
    return f"""You are a technical assistant. Answer using ONLY the provided context.
If the context is insufficient, say so explicitly — do not hallucinate.

<context>
{context_block}
</context>

Question: {query}"""

print(build_rag_prompt(query, retrieve(query)))

You are a technical assistant. Answer using ONLY the provided context.
If the context is insufficient, say so explicitly — do not hallucinate.

<context>
[python_gc_0 | score=0.681]
Python uses reference counting as its primary garbage collection mechanism. Every object has a reference count; when it drops to zero the memory is fr

[python_gc_1 | score=0.642]
drops to zero the memory is freed immediately. Cyclic references (A->B->A) are handled by a separate cyclic GC that runs periodically. The cyclic GC i

[python_gc_3 | score=0.531]
e collected less frequently. You can interact with it via the `gc` module.
</context>

Question: How does Python handle cyclic references?


### Step 5 — Generate
Leveraging rag is now as simple as throwing it back into the LLM

In [7]:
def rag_query(query: str, k: int = 3) -> str:
    chunks = retrieve(query, k=k)
    prompt = build_rag_prompt(query, chunks)
    resp = client.chat.completions.create(
        model=config['big_model'],
        max_tokens=512,
        messages=[
            {"role": "system", "content": "You are a technical assistant."},
            {"role": "user",   "content": prompt},
        ],
    )
    return resp.choices[0].message.content

print(rag_query(query))


### Step 6 — Common failures

1. Query outside corpus
  - Retreival typically always returns results, so threshold the scores or apply some filtering.

In [8]:
# Failure 1: 

ood = "What is the JVM garbage collector pause algorithm?"
for r in retrieve(ood):
    print(f"score={r['score']:.3f}  {r['text'][:80]}")

score=0.617  Go uses a concurrent tri-color mark-and-sweep garbage collector. The GC runs con
score=0.590  ze stop-the-world pauses. Go 1.5 introduced a hybrid barrier that keeps pause ti
score=0.539  e collected less frequently. You can interact with it via the `gc` module.


2. Poor splitting
  - If the relevant context for a statement in your corpus is split between two embedding vectors, you may lose information you are looking for.
  - Simple way to avoid this is have overlap between your splits.
  - Try a different splitting strategy.

In [9]:
tiny = []
for doc_id, text in CORPUS.items():
    for i, chunk in enumerate(chunk_text(text, chunk_size=60, overlap=0)):
        tiny.append({"id": f"{doc_id}_{i}", "text": chunk})
for c in tiny[:6]:
    print(f"'{c['text']}'")
print("\nSentences split mid-thought → retrieval returns incomplete facts.")

'Python uses reference counting as its primary garbage collec'
'tion mechanism. Every object has a reference count; when it '
'drops to zero the memory is freed immediately. Cyclic refere'
'nces (A->B->A) are handled by a separate cyclic GC that runs'
' periodically. The cyclic GC is generational: objects that s'
'urvive collection are promoted to older generations which ar'

Sentences split mid-thought → retrieval returns incomplete facts.


3. Lost-in-the-middle
  - Fixed vectors can't represent infinite sequences.
  - LLMs attend better to START and END of context.
  - Relevant chunks buried in a large window degrade recall.
  - Mitigation: rerank and put highest-score chunks first/last.

In [10]:
large_text = '''Lorem ipsum dolor sit amet, consectetur adipiscing elit. Sed non nunc euismod, condimentum justo sed, viverra ligula. Curabitur interdum, mauris ac facilisis tincidunt, eros lacus egestas arcu, sit amet varius mi velit id lacus. Integer nec velit vitae neque pharetra elementum. Donec euismod, augue sed vestibulum facilisis, arcu nisi finibus neque, sed ultrices ipsum justo a orci. Suspendisse potenti. Praesent vitae mauris et eros finibus interdum.
Pellentesque habitant morbi tristique senectus et netus et malesuada fames ac turpis egestas. Vivamus a diam at urna sagittis cursus. Morbi feugiat, sapien nec varius pharetra, lacus lorem iaculis mi, at efficitur dolor est in risus. Nulla facilisi. Aliquam erat volutpat. Sed eu purus id erat pellentesque bibendum. Maecenas suscipit, arcu sit amet congue cursus, lectus augue scelerisque orci, vel aliquet sapien augue vel tellus.
Sed cursus lectus at turpis luctus, sed posuere lectus pellentesque. Fusce id orci ac elit vulputate mollis. Nam interdum augue a ante pretium, et lacinia ex consequat. Duis ultrices, sem in feugiat fermentum, nibh eros porta arcu, ut mattis leo arcu sit amet libero. Aenean malesuada, magna in tempor efficitur, nunc odio ultricies sapien, sed fermentum odio arcu id lorem. Duis vitae massa ac nisi aliquet tristique.
Nullam vel cursus orci. Nunc in velit eget nulla dignissim faucibus. Proin sit amet ipsum a arcu egestas consequat. Nulla tincidunt, purus non luctus varius, enim massa fringilla enim, id feugiat mi lectus nec metus. Donec id eros posuere, pellentesque odio at, congue ipsum. Donec lacinia, nisl in egestas aliquet, mi massa efficitur sapien, sit amet suscipit mauris mi sed erat.
Ut sed venenatis libero. Vestibulum ante ipsum primis in faucibus orci luctus et ultrices posuere cubilia curae; Cras feugiat, nisl ac scelerisque sagittis, sapien lectus cursus dui, a porta dui urna at risus. Phasellus feugiat lacinia purus, sed pretium neque egestas a. In rhoncus libero vitae ligula malesuada, vitae bibendum libero tincidunt. Etiam tristique, nunc ut interdum congue, eros leo laoreet dolor, sed tempor enim dolor a elit.

There are four lights.

Mauris tempor felis nec arcu pulvinar, sit amet interdum elit commodo. Sed cursus, sem id aliquam ultricies, augue est tincidunt eros, eget vulputate nunc est ut nulla. Nulla et lacus at erat pulvinar ultrices. Suspendisse aliquam ex a urna sodales, in luctus elit feugiat. Integer accumsan ante vitae sapien scelerisque, at aliquet turpis commodo.
Cras varius, sapien sed varius laoreet, arcu enim aliquam dolor, a finibus nisi sem ut purus. Integer vitae risus ut urna imperdiet laoreet. Donec eu lorem id nisl dignissim aliquet. Vivamus viverra ante sed arcu dignissim, vitae cursus mauris elementum. Duis a purus eu nulla condimentum facilisis. Morbi ac mi sit amet magna ultricies dignissim.
Aenean pretium, velit sed luctus fermentum, massa nisl posuere lectus, vel tincidunt sem augue a nisi. Sed sed tincidunt urna. Proin et felis nec arcu rhoncus eleifend. Vestibulum eu interdum enim. Integer venenatis est eu risus laoreet, et accumsan orci molestie. Praesent condimentum, nisl vel facilisis aliquet, nisl dui tincidunt lorem, in tristique massa dolor at erat.
Donec egestas tortor sed sem faucibus, vitae posuere lorem fermentum. Integer tincidunt, est vitae gravida bibendum, dolor ex consequat arcu, a tincidunt turpis orci nec arcu. Vestibulum aliquet nulla a vestibulum scelerisque. Curabitur sed ipsum vitae felis pharetra tincidunt. Praesent at justo sed odio hendrerit varius. Sed luctus, turpis nec consectetur dictum, purus justo bibendum purus, nec finibus elit felis sed elit.
Fusce vitae velit sed lorem ultrices accumsan. Duis non justo a neque efficitur vulputate. Nam porttitor, nibh ac consequat bibendum, elit mi gravida risus, quis pulvinar augue enim eu justo. Sed sed nisi at nibh bibendum ultrices. Quisque ut sem id nisi posuere fermentum. Nam posuere luctus mi, vitae luctus turpis iaculis a.
'''

medium_text = 'There are four lights.'

small_text = 'lights.'

myindex = embed_real([large_text, medium_text, small_text])

q_vec = embed(['How many lights are there?'])[0]
scores = myindex @ q_vec                    # (N,) cosine similarities
top_k = np.argsort(scores)[::-1]
top_k, scores[top_k]

(array([2, 1, 0]), array([0.8264038, 0.8180324, 0.5594896], dtype=float32))

- A lot of techniques exist to try and overcome this, for example:
  - [Cross encoders](https://sbert.net/examples/cross_encoder/applications/README.html)
  - [Hierarchical](https://arxiv.org/abs/2503.10150)/[Graph RAG](https://www.microsoft.com/en-us/research/publication/from-local-to-global-a-graph-rag-approach-to-query-focused-summarization/)
  - [HyDE embeddings](https://arxiv.org/abs/2212.10496)

In [12]:
from sentence_transformers import CrossEncoder

# Load a pre-trained CrossEncoder model
model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Predict scores for a pair of sentences
scores = model.rank('How many lights are there?',
    [large_text, medium_text, small_text])
print(scores)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'corpus_id': 1, 'score': np.float32(7.84675)}, {'corpus_id': 2, 'score': np.float32(-2.6478467)}, {'corpus_id': 0, 'score': np.float32(-10.8942795)}]


## Production Example

https://docs.langchain.com/oss/python/langchain/rag

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
import openai

In [14]:
DEFAULT_CHUNK_SIZE = 4096
DEFAULT_CHUNK_OVERLAP = 512

In [15]:
embedder = OpenAIEmbeddings(check_embedding_ctx_length=False, base_url=config['base_url'], model=config['embedding_model'])
vector_store = InMemoryVectorStore(embedder)

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents([Document(x) for x in CORPUS.values()])

In [17]:
vector_store.add_documents(all_splits)

['55612d20-8e9a-455b-80fe-685d04c1be06',
 'effb8fa9-7e0f-4d82-8c61-718067093575',
 '1ac16f96-3013-44c1-a011-574d677ffeb2']

In [18]:
vector_store.similarity_search('How does Python handle cyclic references?')

[Document(id='55612d20-8e9a-455b-80fe-685d04c1be06', metadata={'start_index': 9}, page_content='Python uses reference counting as its primary garbage collection mechanism.\n        Every object has a reference count; when it drops to zero the memory is freed immediately.\n        Cyclic references (A->B->A) are handled by a separate cyclic GC that runs periodically.\n        The cyclic GC is generational: objects that survive collection are promoted to older generations\n        which are collected less frequently. You can interact with it via the `gc` module.'),
 Document(id='effb8fa9-7e0f-4d82-8c61-718067093575', metadata={'start_index': 9}, page_content='Rust enforces memory safety at compile time through the ownership system.\n        Each value has exactly one owner; ownership can be moved or borrowed.\n        A borrow can be either shared (&T, many readers) or exclusive (&mut T, one writer).\n        These rules are checked by the borrow checker and eliminate use-after-free\n   